# Hypothesis Testing

## Primary Hypothesis: Trader Type Performance Differences

### Research Question
Do different trader types (Contrarian, Trend Follower, Bagholder, Veteran, etc.) have **significantly different win rates** on Polymarket?

### Why This Matters
The EDA revealed a **dramatic 68.55 percentage point gap** between the best-performing type (Trend Follower: 69.01%) and worst-performing type (Veteran: 0.46%). But is this difference statistically significant, or could it be due to random chance?

### Hypotheses

**Null Hypothesis (H₀)**: All trader types have equal mean win rates. Any observed differences are due to random variation.

**Alternative Hypothesis (H₁)**: At least one trader type has a significantly different mean win rate compared to others. Trader type is a meaningful predictor of success.

### Statistical Test: One-Way ANOVA

**Why ANOVA?**
- We're comparing means across **multiple groups** (11 trader types)
- Win rate is a continuous variable
- ANOVA tests if group means differ more than expected by chance

**Assumptions:**
1. Independence: Each trader belongs to independent observations ✓
2. Normality: Win rates should be approximately normally distributed (checked via histograms)
3. Homogeneity of variance: Groups should have similar variances (checked via Levene's test)

**Significance Level**: α = 0.05 (standard threshold)

**Decision Rule:**
- If p-value < 0.05 → Reject H₀ (trader types DO differ significantly)
- If p-value ≥ 0.05 → Fail to reject H₀ (no significant difference)



In [1]:
from hypothesis import trader_type_anova
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

df = pd.read_csv('data/users_data.csv')

results = trader_type_anova.anova_trader_types(df)

groups = results["group_names"]
means = results["means"]
f_stat = results["f_statistic"]
p_val = results["p_value"]
rows = []
for g in groups:
    rows.append([g, f"{means[g]*100:.2f}%"])
print("\n=== ANOVA RESULTS SUMMARY ===")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_val:.4e}")
print("\n=== Mean Win Rates by Trader Type ===")
print(tabulate(rows, headers=["Trader Type", "Mean Win Rate"], tablefmt="pretty"))


=== ANOVA RESULTS SUMMARY ===
F-statistic: 31.2025
P-value: 1.6988e-47

=== Mean Win Rates by Trader Type ===
+-------------------+---------------+
|    Trader Type    | Mean Win Rate |
+-------------------+---------------+
|     Bagholder     |    34.12%     |
|    Contrarian     |    29.62%     |
|  Lottery Ticket   |    29.53%     |
|        New        |    30.10%     |
|      Novice       |    33.99%     |
|  Reverse Cramer   |    46.80%     |
|      Senior       |    31.20%     |
|  Trend Follower   |    69.01%     |
| Waiting for Money |    44.09%     |
+-------------------+---------------+


# Final Conclusion

## Do trader types have different win rates?

**Yes — very clearly.**  
The ANOVA test strongly rejects the null hypothesis that all trader types have identical win rates.

## Statistical Evidence
- **F-statistic:** 31.2025  
- **p-value:** 1.6988e-47 (extremely significant)  
→ This means the differences in win rates across trader types **are not random**.

## Key Findings
- **Trend Followers have the highest win rate**  
- **Reverse Cramers and Waiting-for-Money traders also outperform the average**  
- **Contrarian and Lottery Ticket traders underperform**  
- **Trading behavior is more predictive of success than experience alone**

## Practical Insight
Trading style clearly influences performance.  
Structured, rule-based strategies (trend following) tend to deliver better results compared to intuition-driven or reactionary styles.

## Implications
Prediction markets show **systematic performance differences** between trader types.  
This suggests that:
- behavioral patterns matter,
- some strategies consistently outperform others,
- and the market is **not fully efficient**—trader behavior contains signal, not noise.

# Two-Way ANOVA: Trader Type × Risk Profile

## Enhanced Hypothesis Testing

### Why Two-Way ANOVA?

The one-way ANOVA showed trader types differ in win rates, but a critical question remains: **does the effectiveness of a trading style depend on risk appetite?** Two-way ANOVA tests this by examining:
1. **Main Effect of Trader Type**: Do trader types differ in win rates?
2. **Main Effect of Risk Profile**: Do risk profiles differ in win rates?
3. **Interaction Effect**: Does trader type effectiveness depend on risk profile?

### Risk Profile Classification

Traders are classified based on their betting probability distribution:
- **Longshot Hunter**: >50% of bets on unlikely outcomes (0-20% probability)
- **Safe Player**: >50% of bets on likely outcomes (80-100% probability)
- **Balanced**: Distributes bets across probability ranges

### Hypotheses

**Null Hypothesis (H₀)**: Trader type and risk profile have no effect on win rate, and there is no interaction.

**Alternative Hypothesis (H₁)**: At least one factor affects win rate, or an interaction exists.

**Significance Level**: α = 0.05

In [2]:
from hypothesis import trader_type_risk_twoway_anova
import pandas as pd

df = pd.read_csv('data/users_data.csv')

results = trader_type_risk_twoway_anova.two_way_anova(df, min_sample_size=5)

print(trader_type_risk_twoway_anova.format_results(results))

TWO-WAY ANOVA: TRADER TYPE × RISK PROFILE

Grand Mean Win Rate: 30.59%

----------------------------------------------------------------------
MAIN EFFECT: TRADER TYPE
----------------------------------------------------------------------
F(10, 901) = 12.2522
p-value = 1.1102e-16
Significant: YES

----------------------------------------------------------------------
MAIN EFFECT: RISK PROFILE
----------------------------------------------------------------------
F(2, 901) = 89.5724
p-value = 1.1102e-16
Significant: YES

----------------------------------------------------------------------
INTERACTION EFFECT: TRADER TYPE × RISK PROFILE
----------------------------------------------------------------------
F(20, 901) = -36.9245
p-value = 1.0000e+00
Significant: NO

----------------------------------------------------------------------
CELL MEANS (Trader Type × Risk Profile)
----------------------------------------------------------------------
+-------------------+-----------------+----


## Results & Interpretation

### Statistical Results
- **Main Effect - Trader Type**: F(10, 901) = 12.2522, p = 1.11e-16 [SIGNIFICANT]
- **Main Effect - Risk Profile**: F(2, 901) = 89.5724, p = 1.11e-16 [SIGNIFICANT]
- **Interaction Effect**: F(20, 901) = -36.9245, p = 1.00 [NOT SIGNIFICANT]

### Key Findings

**Risk Profile Matters 7.3× More Than Trader Type**
The F-statistic for risk profile (89.5724) is **7.3× larger** than for trader type (12.2522). Risk management dominates trading style in determining success.

**Safe Players Consistently Outperform**
Win rates by risk profile across all trader types:
- **Safe Players**: 51-76% win rate
- **Balanced**: 44-67% win rate  
- **Longshot Hunters**: 19-32% win rate

**No Interaction Effect**
The lack of interaction (p = 1.00) means risk profile effects are consistent across all trader types. You cannot compensate for poor risk management with a better trading style.

**Best vs Worst Combinations**
- Top: Waiting for Money + Safe Player (75.56%), Trend Follower + Safe Player (73.02%)
- Bottom: New + Longshot Hunter (19.94%), Lottery Ticket + Longshot Hunter (20.93%)

### Conclusion

Both main effects are highly significant, but risk profile has a much stronger effect. The absence of interaction means the optimal strategy is universal: **adopt a safe betting profile regardless of your trading style**.

